In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from pyramid_train import BATCH, MODEL_PATH, TemporalPyramid, build_cloud_day, load_model, pool_frame, train_and_save

def main(datasources, start_date, end_date):
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f'Missing model: {MODEL_PATH}')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = load_model(MODEL_PATH, map_location=device)
    stats = (np.asarray(checkpoint['mean'], np.float32), np.asarray(checkpoint['std'], np.float32))
    model = TemporalPyramid(input_dim=len(checkpoint['feature_cols'])).to(device)
    model.load_state_dict(checkpoint['state_dict']); model.eval()
    pool = pool_frame(start_date, end_date); scored = []
    with torch.no_grad():
        for day, day_pool in pool.groupby('date', sort=True):
            X, index = build_cloud_day(datasources['bar1m'], day, day_pool['instrument'].tolist(), stats)
            tensor = torch.from_numpy(X); predictions = []
            for offset in range(0, len(index), BATCH):
                predictions.append(model(tensor[offset:offset+BATCH].to(device)).cpu().numpy())
            index['score'] = np.concatenate(predictions).astype(np.float64); scored.append(index)
    result = (pd.concat(scored, ignore_index=True).replace([np.inf, -np.inf], np.nan)
              .dropna(subset=['score']).drop_duplicates(['date','instrument'])
              [['date','instrument','score']].sort_values(['date','instrument']).reset_index(drop=True))
    if result.empty: raise RuntimeError('Inference produced no valid scores')
    return result
